# CNN + BiLSTM + Attention Gesture Classifier

## Architecture Rationale

This notebook implements a three-stage hybrid architecture for glove-based gesture recognition:

### Stage 1 — Convolutional Frontend (Spatial / Local Feature Extraction)
1D Conv layers slide a kernel along the **time axis** across all sensor channels simultaneously.  
A short kernel (size 5) captures **local temporal patterns** — the micro-movements and co-activations between sensors that distinguish one gesture phase from another.  
Stacking two Conv1D blocks (with increasing filter counts and max-pooling) compresses time while building progressively abstract representations.

### Stage 2 — Bidirectional LSTM (Sequential Context)
Standard LSTM processes a sequence **left-to-right** only. It therefore relies entirely on what has already happened when predicting the next hidden state, and has no knowledge of future context.

**Bidirectional LSTM (BiLSTM)** runs two LSTM cells in parallel:
- **Forward cell**: left-to-right (causal context — "what happened before")
- **Backward cell**: right-to-left (anti-causal context — "what comes after")

Their hidden states are concatenated at each timestep.  
For offline gesture recognition on a **fixed-length window**, this is strictly better than a unidirectional LSTM: knowing that the hand will end in a fist helps interpret an ambiguous mid-gesture position.  
The tradeoff is doubled parameter count and non-applicability to real-time streaming (where future timesteps are unknown).

### Stage 3 — Attention Mechanism (Temporal Focus)
Not every timestep carries equal discriminative information.  
The peak of a finger-bend, the moment of maximum wrist rotation — these brief events define the gesture.  
An **attention layer** learns a scalar weight for each timestep and computes a weighted sum over the BiLSTM output sequence, producing a single **context vector** that emphasises the most gesture-relevant moments.  
This also provides interpretability: plotting the attention weights shows *where in time* the model looked when making its decision.

Two attention variants are implemented (selectable via `ATTENTION_TYPE`):
- **Additive / Bahdanau**: single-head, custom Keras layer — matches the A-CBLN paper style
- **Scaled dot-product / Multi-head**: uses `tf.keras.layers.MultiHeadAttention` — transformer-style, supports parallel heads

---

## Paper Reference — A-CBLN

Wu et al. (2023) — *"Data glove-based gesture recognition using CNN-BiLSTM model with attention mechanism"*  
**A-CBLN architecture**: 2D CNN (kernel 1×3, 16→32 filters) → 2× BiLSTM (8 units) → Attention → Dense 16 → softmax  
Achieved **95.05% accuracy** on a 7-class handwashing gesture dataset.  
Adam lr=0.001, batch_size=64, epochs=50.

This notebook uses an upgraded variant (larger CNN/BiLSTM capacity, configurable attention) suited for the higher-dimensional glove sensor stream.

---

## Data Convention
Column naming: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`, `right_palm_az`

Quaternion columns (`qw`, `qx`, `qy`, `qz`) are always excluded.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

packages = [
    "tensorflow",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# Optional: graphviz for plot_model
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydot"], check=False)
subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=False)

print("Dependencies ready.")

In [ ]:
# Cell 2 — Imports
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input, backend as K
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, BatchNormalization, Dropout,
    Bidirectional, LSTM, Dense, Flatten, GlobalAveragePooling1D,
    Multiply, Lambda, Permute, RepeatVector, Activation,
    MultiHeadAttention, LayerNormalization,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import plot_model

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# ── Project utilities ─────────────────────────────────────────────────────────
UTILS_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'utils')
if UTILS_PATH not in sys.path:
    sys.path.insert(0, UTILS_PATH)

from data_loader import build_dataset, split_dataset, build_column_groups

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras    version   : {keras.__version__}")
print(f"NumPy    version   : {np.__version__}")
print(f"GPU devices        : {tf.config.list_physical_devices('GPU')}")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Cell 3 — Configuration

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'
FS_HZ     = 22.0

# ── COLUMN SELECTION ──────────────────────────────────────────────────────────
USE_LEFT_HAND     = True
USE_RIGHT_HAND    = True
USE_DISTAL_IMU    = True
USE_PROXIMAL_IMU  = True
USE_ACCELEROMETER = True
USE_ORIENTATION   = True
USE_FLEX_SENSORS  = True
USE_PALM_IMU      = True
USE_WRIST_IMU     = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
FILTER_TYPE     = 'butterworth_lp'
TARGET_LEN      = 110
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# CNN frontend (1D convolution along time axis)
CNN_FILTERS     = [64, 128]
CNN_KERNEL_SIZE = 5
CNN_POOL_SIZE   = 2
CNN_DROPOUT     = 0.2

# BiLSTM
BILSTM_UNITS       = [64, 32]   # Units per BiLSTM layer (each layer is bidirectional)
BILSTM_DROPOUT     = 0.3
BILSTM_REC_DROPOUT = 0.2

# Attention
ATTENTION_TYPE  = 'additive'   # 'additive' (Bahdanau) | 'scaled_dot' (transformer-style)
ATTENTION_HEADS = 4            # For multi-head scaled_dot attention

# Dense head
DENSE_UNITS  = [64]
DROPOUT_RATE = 0.3

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 60
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 10

# ── OUTPUT PATHS ──────────────────────────────────────────────────────────────
MODEL_DIR   = 'saved_models'
RESULTS_DIR = 'results'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  Attention type : {ATTENTION_TYPE}")
print(f"  CNN filters    : {CNN_FILTERS}  kernel {CNN_KERNEL_SIZE}")
print(f"  BiLSTM units   : {BILSTM_UNITS}")
print(f"  Dense units    : {DENSE_UNITS}")
print(f"  Epochs/batch   : {EPOCHS} / {BATCH_SIZE}")

In [ ]:
# Cell 4 — Build feature columns from config

# Map config flags to build_column_groups parameters
active_hands = []
if USE_LEFT_HAND:  active_hands.append('left')
if USE_RIGHT_HAND: active_hands.append('right')

# Collect active locations
active_locs = []
if USE_DISTAL_IMU:   active_locs.append('mid')
if USE_PROXIMAL_IMU: active_locs.append('prox')

# Build column groups using the utility
col_groups = build_column_groups(
    hands        = active_hands  or None,
    locs         = active_locs   or None,
    include_imu  = USE_ACCELEROMETER or USE_ORIENTATION,
    include_flex = USE_FLEX_SENSORS,
    include_palm = USE_PALM_IMU,
    include_wrist= USE_WRIST_IMU,
)

# Flatten all group columns into a single feature list
all_feature_cols = []
for group, cols in col_groups.items():
    for col in cols:
        # Filter sub-channels based on config
        is_accel  = any(col.endswith(f'_{c}') for c in ['ax','ay','az'])
        is_orient = any(col.endswith(f'_{c}') for c in ['pitch','roll','yaw'])
        # Flex sensors don't carry ax/ay/az/pitch/roll/yaw suffixes
        is_flex   = 'flex' in col

        if is_flex:
            if USE_FLEX_SENSORS:
                all_feature_cols.append(col)
        elif is_accel and USE_ACCELEROMETER:
            all_feature_cols.append(col)
        elif is_orient and USE_ORIENTATION:
            all_feature_cols.append(col)

# Deduplicate while preserving order
seen = set()
feature_cols = []
for c in all_feature_cols:
    if c not in seen:
        seen.add(c)
        feature_cols.append(c)

print(f"Feature columns selected : {len(feature_cols)}")
print(f"Groups                   : {list(col_groups.keys())[:6]} ... ({len(col_groups)} total)")
print(f"\nFirst 20 columns:")
for i, c in enumerate(feature_cols[:20]):
    print(f"  {i+1:>3}. {c}")
if len(feature_cols) > 20:
    print(f"  ... ({len(feature_cols) - 20} more)")

In [ ]:
# Cell 5 — Load dataset & class distribution

print("Loading dataset from:", os.path.abspath(DATA_ROOT))
print("─" * 60)

X, y, LABELS, feature_cols_used = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols if len(feature_cols) > 0 else None,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
    exclude_quat    = True,
)

N_CLASSES  = len(LABELS)
N_FEATURES = X.shape[2]
print(f"\nFinal shapes  → X: {X.shape}, y: {y.shape}")
print(f"Classes ({N_CLASSES}): {LABELS}")
print(f"Features used: {N_FEATURES}")

# ── Class distribution plot ───────────────────────────────────────────────────
counts = pd.Series(y).map(dict(enumerate(LABELS))).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
axes[0].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Class Distribution (sample counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Gesture class')
axes[0].set_ylabel('Number of samples')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            startangle=140, colors=plt.cm.tab10.colors[:len(counts)])
axes[1].set_title('Class Balance (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── Sample signal visualisation ───────────────────────────────────────────────
fig, axes = plt.subplots(min(N_CLASSES, 4), 1, figsize=(14, 3 * min(N_CLASSES, 4)), sharex=True)
axes = np.array(axes).flatten()
for ci in range(min(N_CLASSES, 4)):
    idx = np.where(y == ci)[0][0]
    axes[ci].plot(X[idx, :, :6], alpha=0.7, linewidth=0.9)
    axes[ci].set_title(f'Class: {LABELS[ci]}  (first 6 channels)', fontsize=10)
    axes[ci].set_ylabel('Norm. value')
axes[-1].set_xlabel('Time step')
plt.suptitle('Sample Gesture Signals (one example per class)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_sample_signals.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Dataset split

(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

print("\nSplit summary:")
print(f"  Train : {X_train.shape}  labels {y_train.shape}")
print(f"  Val   : {X_val.shape}  labels {y_val.shape}")
print(f"  Test  : {X_test.shape}  labels {y_test.shape}")

# Convert labels to one-hot for training
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)

# Per-split class balance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ys, title in zip(axes,
                         [y_train, y_val, y_test],
                         ['Train', 'Validation', 'Test']):
    counts_s = pd.Series(ys).value_counts().sort_index()
    ax.bar([LABELS[i] for i in counts_s.index], counts_s.values, color='steelblue', edgecolor='white')
    ax.set_title(f'{title} set — {len(ys)} samples', fontsize=10, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_split_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Model definition: CNN + BiLSTM + Attention
# ─────────────────────────────────────────────────────────────────────────────
# Uses the Keras Functional API throughout (not Sequential) so that the
# attention layer can branch off the BiLSTM sequence and re-merge.
# ─────────────────────────────────────────────────────────────────────────────

# ── Custom Additive (Bahdanau) Attention Layer ────────────────────────────────
class BahdanauAttention(layers.Layer):
    """
    Additive attention (Bahdanau, 2015) over a temporal sequence.

    Input  : (batch, T, H)  — BiLSTM output sequence
    Output : (batch, H)     — context vector  (weighted sum over T)
             (batch, T)     — attention weights (for visualisation)

    Score function:
        e_t  = v^T · tanh(W · h_t + b)
        α_t  = softmax(e_t)
        c    = Σ_t  α_t · h_t
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W = layers.Dense(units, use_bias=True)
        self.V = layers.Dense(1,     use_bias=False)

    def call(self, hidden_seq, training=False):
        # hidden_seq: (batch, T, H)
        score  = self.V(tf.nn.tanh(self.W(hidden_seq)))  # (batch, T, 1)
        alpha  = tf.nn.softmax(score, axis=1)             # (batch, T, 1)
        context = tf.reduce_sum(alpha * hidden_seq, axis=1)  # (batch, H)
        return context, tf.squeeze(alpha, axis=-1)           # context, weights

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


# ── Model builder ─────────────────────────────────────────────────────────────
def build_cnn_bilstm_attention(
        seq_len        = TARGET_LEN,
        n_features     = N_FEATURES,
        n_classes      = N_CLASSES,
        cnn_filters    = CNN_FILTERS,
        cnn_kernel     = CNN_KERNEL_SIZE,
        cnn_pool       = CNN_POOL_SIZE,
        cnn_dropout    = CNN_DROPOUT,
        bilstm_units   = BILSTM_UNITS,
        bilstm_drop    = BILSTM_DROPOUT,
        bilstm_rec_drop= BILSTM_REC_DROPOUT,
        attention_type = ATTENTION_TYPE,
        attention_heads= ATTENTION_HEADS,
        dense_units    = DENSE_UNITS,
        dropout_rate   = DROPOUT_RATE,
        learning_rate  = LEARNING_RATE,
):
    """
    Build and compile the CNN + BiLSTM + Attention model.

    Returns a compiled tf.keras.Model.
    """
    # ── Input ─────────────────────────────────────────────────────────────
    inp = Input(shape=(seq_len, n_features), name='sensor_input')

    # ── CNN frontend ──────────────────────────────────────────────────────
    x = inp
    for i, filters in enumerate(cnn_filters):
        x = Conv1D(
            filters     = filters,
            kernel_size = cnn_kernel,
            padding     = 'same',
            activation  = 'relu',
            name        = f'conv1d_{i+1}',
        )(x)
        x = BatchNormalization(name=f'bn_conv_{i+1}')(x)
        x = MaxPooling1D(pool_size=cnn_pool, name=f'pool_{i+1}')(x)
        x = Dropout(cnn_dropout, name=f'drop_conv_{i+1}')(x)

    # After CNN: shape = (batch, seq_len // pool^n_layers, cnn_filters[-1])
    cnn_out = x

    # ── BiLSTM stack ──────────────────────────────────────────────────────
    for j, units in enumerate(bilstm_units):
        return_seq = True  # must return sequences for all layers (attention needs them)
        x = Bidirectional(
            LSTM(
                units,
                return_sequences = return_seq,
                dropout          = bilstm_drop,
                recurrent_dropout= bilstm_rec_drop,
            ),
            merge_mode = 'concat',
            name       = f'bilstm_{j+1}',
        )(x)
        x = BatchNormalization(name=f'bn_lstm_{j+1}')(x)

    bilstm_out = x  # (batch, T', 2 * bilstm_units[-1])

    # ── Attention layer ───────────────────────────────────────────────────
    attention_type = attention_type.lower()

    if attention_type == 'additive':
        # Bahdanau single-head additive attention
        attn_layer = BahdanauAttention(units=bilstm_units[-1] * 2, name='bahdanau_attention')
        context, attn_weights = attn_layer(bilstm_out)
        # context: (batch, 2*bilstm_units[-1])

    elif attention_type == 'scaled_dot':
        # Multi-head scaled dot-product attention (transformer-style)
        # Query = Key = Value = BiLSTM output sequence
        mha = MultiHeadAttention(
            num_heads   = attention_heads,
            key_dim     = bilstm_units[-1] * 2 // attention_heads,
            name        = 'multihead_attention',
        )
        attn_out, attn_scores = mha(
            query         = bilstm_out,
            value         = bilstm_out,
            key           = bilstm_out,
            return_attention_scores = True,
        )
        # Residual + layer-norm (transformer convention)
        attn_out = LayerNormalization(name='attn_layer_norm')(attn_out + bilstm_out)
        # Pool over time → context vector
        context = GlobalAveragePooling1D(name='global_avg_pool')(attn_out)
        # Average attention weights across heads and queries for visualisation
        # attn_scores: (batch, heads, T_q, T_k)  →  mean over heads + queries
        attn_weights = tf.reduce_mean(attn_scores, axis=[1, 2])  # (batch, T_k)

    else:
        raise ValueError(f"Unknown attention_type: '{attention_type}'. "
                         "Choose 'additive' or 'scaled_dot'.")

    # ── Dense head ────────────────────────────────────────────────────────
    x = context
    for k, units in enumerate(dense_units):
        x = Dense(units, activation='relu', name=f'dense_{k+1}')(x)
        x = Dropout(dropout_rate, name=f'drop_dense_{k+1}')(x)

    output = Dense(n_classes, activation='softmax', name='output')(x)

    # ── Assemble model ────────────────────────────────────────────────────
    # We expose attn_weights as a named output for easy extraction later
    model = Model(
        inputs  = inp,
        outputs = [output, attn_weights],
        name    = f'CNN_BiLSTM_{attention_type.capitalize()}Attention',
    )

    # Compile (train only the primary output)
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = {'output': 'categorical_crossentropy'},
        metrics   = {'output': ['accuracy']},
    )
    return model


# ── Instantiate & summarise ───────────────────────────────────────────────────
model = build_cnn_bilstm_attention()
model.summary()

# Architecture diagram (requires graphviz)
try:
    diagram_path = os.path.join(RESULTS_DIR, '04_model_architecture.png')
    plot_model(
        model,
        to_file    = diagram_path,
        show_shapes= True,
        show_dtype = False,
        show_layer_names = True,
        expand_nested    = True,
        dpi=96,
    )
    from IPython.display import Image, display
    display(Image(diagram_path))
    print(f"Model diagram saved to {diagram_path}")
except Exception as e:
    print(f"[plot_model skipped] — graphviz not installed: {e}")

# Parameter count summary
total_params     = model.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

In [ ]:
# Cell 8 — Training + learning curves

checkpoint_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attn_best.keras')

callbacks = [
    EarlyStopping(
        monitor              = 'val_output_accuracy',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor  = 'val_output_loss',
        factor   = 0.5,
        patience = 5,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
    ModelCheckpoint(
        filepath          = checkpoint_path,
        monitor           = 'val_output_accuracy',
        save_best_only    = True,
        save_weights_only = False,
        verbose           = 0,
    ),
]

print(f"Training CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention model")
print(f"  Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print("─" * 60)

history = model.fit(
    X_train, {'output': y_train_oh},
    validation_data = (X_val, {'output': y_val_oh}),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

# ── Learning curves ───────────────────────────────────────────────────────────
hist = history.history

# Determine correct metric keys
acc_key     = 'output_accuracy'     if 'output_accuracy'     in hist else 'accuracy'
val_acc_key = 'val_output_accuracy' if 'val_output_accuracy' in hist else 'val_accuracy'
loss_key    = 'output_loss'         if 'output_loss'         in hist else 'loss'
val_loss_key= 'val_output_loss'     if 'val_output_loss'     in hist else 'val_loss'

epochs_ran = range(1, len(hist[loss_key]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(epochs_ran, hist[acc_key],     label='Train accuracy', linewidth=2)
ax1.plot(epochs_ran, hist[val_acc_key], label='Val accuracy',   linewidth=2, linestyle='--')
best_val_acc = max(hist[val_acc_key])
best_epoch   = np.argmax(hist[val_acc_key]) + 1
ax1.axvline(best_epoch, color='red', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax1.set_title(f'Accuracy  |  Best val: {best_val_acc:.4f}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(epochs_ran, hist[loss_key],     label='Train loss', linewidth=2)
ax2.plot(epochs_ran, hist[val_loss_key], label='Val loss',   linewidth=2, linestyle='--')
ax2.axvline(best_epoch, color='red', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax2.set_title('Cross-Entropy Loss', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention — Training Curves',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f"\nBest validation accuracy : {best_val_acc:.4f}  at epoch {best_epoch}")
print(f"Model checkpoint saved   : {checkpoint_path}")

In [ ]:
# Cell 9 — Evaluation: classification report + confusion matrix

# ── Predictions ───────────────────────────────────────────────────────────────
pred_outputs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
# Model has two outputs: [class_probs, attn_weights]
if isinstance(pred_outputs, (list, tuple)):
    y_prob      = pred_outputs[0]   # (N_test, N_classes)
    attn_test   = pred_outputs[1]   # (N_test, T)
else:
    y_prob    = pred_outputs
    attn_test = None

y_pred = np.argmax(y_prob, axis=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(
    X_test,
    {'output': y_test_oh},
    batch_size = BATCH_SIZE,
    verbose    = 0,
)[:2]
print(f"Test accuracy : {test_acc:.4f}  |  Test loss : {test_loss:.4f}")

print("\n" + "─" * 60)
print("Classification Report:")
print("─" * 60)
report_str = classification_report(y_test, y_pred, target_names=LABELS, digits=4)
print(report_str)

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, ax=axes[0],
)
axes[0].set_title('Confusion Matrix (counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=35)
axes[0].tick_params(axis='y', rotation=0)

# Normalised
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, vmin=0, vmax=1, ax=axes[1],
)
axes[1].set_title('Confusion Matrix (row-normalised)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=35)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle(f'CNN + BiLSTM + {ATTENTION_TYPE.capitalize()} Attention — Test Set Evaluation',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── Per-class accuracy bar chart ──────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()
colours = ['#d73027' if v < 0.7 else ('#fee090' if v < 0.9 else '#1a9850')
           for v in per_class_acc]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(LABELS, per_class_acc * 100, color=colours, edgecolor='white')
ax.axhline(100 * per_class_acc.mean(), color='steelblue', linestyle='--',
           linewidth=1.5, label=f'Mean {100*per_class_acc.mean():.1f}%')
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=35)
ax.legend()
for bar, v in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{v*100:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_per_class_accuracy.png'), dpi=120, bbox_inches='tight')
plt.show()

## Cell 10 — Attention Weight Visualisation

The attention layer assigns a weight $\alpha_t$ to every timestep $t$ in the gesture window.  
Plotting these weights reveals **which part of the 5-second recording** was most discriminative for each prediction:

- A high weight early in the sequence suggests the **onset** of the gesture carries most information.
- A high weight at the end suggests the gesture's **release/return phase** is decisive.
- Spread weights indicate the model uses the **full trajectory** rather than a single key event.

This interpretability is one of the main motivations for adding the attention mechanism on top of BiLSTM.

In [ ]:
# Cell 10 — Attention weight visualisation

# ── Helper: extract attention weights for a batch ────────────────────────────
def get_attention_weights(model, X_batch):
    """
    Run a forward pass and return:
        y_prob   : (N, C)  — softmax class probabilities
        attn_w   : (N, T)  — per-timestep attention weights
    Works for both additive and scaled_dot attention types since both
    expose attn_weights as the second model output.
    """
    outputs = model.predict(X_batch, verbose=0)
    if isinstance(outputs, (list, tuple)):
        return outputs[0], outputs[1]
    return outputs, None


# ── Select one test sample per class ─────────────────────────────────────────
n_viz       = min(N_CLASSES, 6)   # visualise at most 6 classes
sample_idxs = []
for ci in range(n_viz):
    candidates = np.where(y_test == ci)[0]
    if len(candidates) == 0:
        continue
    # Prefer correctly classified samples for the main plot
    correct = candidates[y_pred[candidates] == ci]
    idx     = correct[0] if len(correct) > 0 else candidates[0]
    sample_idxs.append(idx)

X_viz   = X_test[sample_idxs]                 # (n_viz, T, C)
y_true_viz  = y_test[sample_idxs]
probs_viz, attn_viz = get_attention_weights(model, X_viz)
y_pred_viz  = np.argmax(probs_viz, axis=1)

# ── Attention weight range varies by attention type ───────────────────────────
# For additive: attn_viz shape = (N, T_bilstm)  (timesteps after CNN pooling)
# For scaled_dot: same, averaged over heads
# Upsample back to TARGET_LEN for overlay on original signal
from scipy.interpolate import interp1d as sci_interp

def upsample_weights(w, target_len):
    """Linearly interpolate a 1-D weight vector to target_len."""
    if len(w) == target_len:
        return w
    x_old = np.linspace(0, 1, len(w))
    x_new = np.linspace(0, 1, target_len)
    return sci_interp(x_old, w, kind='linear')(x_new)


# ── Main visualisation: signal + attention overlay ────────────────────────────
fig, axes = plt.subplots(n_viz, 1, figsize=(14, 3.2 * n_viz), sharex=False)
axes = np.array(axes).flatten()

time_axis = np.linspace(0, TARGET_LEN / FS_HZ, TARGET_LEN)

for i, (ax, idx) in enumerate(zip(axes, sample_idxs)):
    signal   = X_test[idx]             # (T, C)
    true_lbl = LABELS[y_true_viz[i]]
    pred_lbl = LABELS[y_pred_viz[i]]
    correct  = '✓' if y_true_viz[i] == y_pred_viz[i] else '✗'

    # Plot sensor channels (first 6, thin + transparent)
    for ch in range(min(signal.shape[1], 6)):
        ax.plot(time_axis, signal[:, ch], alpha=0.35, linewidth=0.8, color='steelblue')

    # Overlay attention weights as a filled area (second y-axis)
    if attn_viz is not None:
        raw_w    = attn_viz[i]                      # (T_attn,)
        w_full   = upsample_weights(raw_w, TARGET_LEN)  # (T,)
        ax2      = ax.twinx()
        ax2.fill_between(time_axis, w_full, alpha=0.3, color='tomato', label='Attention')
        ax2.plot(time_axis, w_full, color='tomato', linewidth=1.2, alpha=0.7)
        ax2.set_ylabel('Attn weight', color='tomato', fontsize=8)
        ax2.tick_params(axis='y', labelcolor='tomato', labelsize=7)
        ax2.set_ylim(0, max(w_full.max() * 1.4, 1e-6))

    ax.set_title(
        f'{correct} True: {true_lbl}  |  Predicted: {pred_lbl}  '
        f'(conf: {probs_viz[i, y_pred_viz[i]]:.2f})',
        fontsize=10,
        color='darkgreen' if correct == '✓' else 'firebrick',
    )
    ax.set_ylabel('Norm. signal')
    ax.set_xlabel('Time (s)')
    ax.grid(True, alpha=0.2)

plt.suptitle(
    f'Attention Weights Overlaid on Sensor Signals  ({ATTENTION_TYPE} attention)\n'
    'Red shaded region = timesteps most influential for prediction',
    fontsize=12, fontweight='bold', y=1.005,
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_attention_visualisation.png'), dpi=120, bbox_inches='tight')
plt.show()


# ── Heatmap: all test samples, attention vs class ─────────────────────────────
if attn_test is not None:
    print("\nAttention weight heatmap (mean per class):")
    # attn_test: (N_test, T_attn)
    T_attn = attn_test.shape[1]
    mean_attn = np.zeros((N_CLASSES, T_attn))
    for ci in range(N_CLASSES):
        ci_idx = np.where(y_test == ci)[0]
        if len(ci_idx) > 0:
            mean_attn[ci] = attn_test[ci_idx].mean(axis=0)

    # Upsample columns to TARGET_LEN for a consistent x-axis
    mean_attn_full = np.stack(
        [upsample_weights(mean_attn[ci], TARGET_LEN) for ci in range(N_CLASSES)], axis=0
    )

    fig, ax = plt.subplots(figsize=(14, max(3, N_CLASSES * 0.8)))
    sns.heatmap(
        mean_attn_full,
        ax=ax,
        cmap='YlOrRd',
        xticklabels=np.round(np.linspace(0, TARGET_LEN / FS_HZ, TARGET_LEN), 1)[::10],
        yticklabels=LABELS,
        cbar_kws={'label': 'Mean attention weight'},
    )
    # Fix x-tick positions
    ax.set_xticks(np.arange(0, TARGET_LEN, 10))
    ax.set_xticklabels(
        [f"{v:.1f}s" for v in np.linspace(0, TARGET_LEN / FS_HZ, len(np.arange(0, TARGET_LEN, 10)))],
        rotation=45, fontsize=8,
    )
    ax.set_title(
        f'Mean Attention Weights per Class (test set, {ATTENTION_TYPE} attention)',
        fontsize=12, fontweight='bold',
    )
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Gesture class')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '04_attention_heatmap.png'), dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 11 — Save model and results JSON

import datetime

# ── Save final (best-weights) model ──────────────────────────────────────────
final_model_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attention_final.keras')
model.save(final_model_path)
print(f"Model saved       : {final_model_path}")

# Also save as TF SavedModel format for TFLite conversion
savedmodel_path = os.path.join(MODEL_DIR, '04_cnn_bilstm_attention_savedmodel')
model.export(savedmodel_path)
print(f"SavedModel saved  : {savedmodel_path}")

# ── Compute final test metrics ────────────────────────────────────────────────
report_dict = classification_report(
    y_test, y_pred, target_names=LABELS, output_dict=True
)

# ── Results JSON ─────────────────────────────────────────────────────────────
results = {
    "notebook"       : "04_CNN_BiLSTM_Attention",
    "timestamp"      : datetime.datetime.now().isoformat(),
    "architecture"   : {
        "type"           : "CNN + BiLSTM + Attention",
        "attention_type" : ATTENTION_TYPE,
        "cnn_filters"    : CNN_FILTERS,
        "cnn_kernel_size": CNN_KERNEL_SIZE,
        "cnn_pool_size"  : CNN_POOL_SIZE,
        "bilstm_units"   : BILSTM_UNITS,
        "dense_units"    : DENSE_UNITS,
        "total_params"   : int(model.count_params()),
    },
    "dataset"        : {
        "n_samples"      : int(X.shape[0]),
        "n_timesteps"    : int(TARGET_LEN),
        "n_features"     : int(N_FEATURES),
        "n_classes"      : int(N_CLASSES),
        "classes"        : LABELS,
        "filter_type"    : FILTER_TYPE,
        "normalization"  : NORMALIZATION,
        "fs_hz"          : FS_HZ,
    },
    "training"       : {
        "epochs_run"     : len(history.history[loss_key]),
        "epochs_config"  : EPOCHS,
        "batch_size"     : BATCH_SIZE,
        "learning_rate"  : LEARNING_RATE,
        "early_stop"     : EARLY_STOP_PATIENCE,
        "best_val_acc"   : float(max(history.history[val_acc_key])),
        "best_epoch"     : int(np.argmax(history.history[val_acc_key]) + 1),
    },
    "evaluation"     : {
        "test_accuracy"  : float(test_acc),
        "test_loss"      : float(test_loss),
        "per_class"      : {
            lbl: {
                "precision" : float(report_dict[lbl]["precision"]),
                "recall"    : float(report_dict[lbl]["recall"]),
                "f1-score"  : float(report_dict[lbl]["f1-score"]),
                "support"   : int(report_dict[lbl]["support"]),
            }
            for lbl in LABELS if lbl in report_dict
        },
        "macro_avg"      : report_dict.get("macro avg", {}),
        "weighted_avg"   : report_dict.get("weighted avg", {}),
    },
    "artifacts"      : {
        "model_keras"    : final_model_path,
        "model_savedmodel": savedmodel_path,
        "class_distribution" : os.path.join(RESULTS_DIR, '04_class_distribution.png'),
        "training_curves"    : os.path.join(RESULTS_DIR, '04_training_curves.png'),
        "confusion_matrix"   : os.path.join(RESULTS_DIR, '04_confusion_matrix.png'),
        "attention_viz"      : os.path.join(RESULTS_DIR, '04_attention_visualisation.png'),
        "attention_heatmap"  : os.path.join(RESULTS_DIR, '04_attention_heatmap.png'),
    },
}

results_path = os.path.join(RESULTS_DIR, '04_cnn_bilstm_attention_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results JSON saved: {results_path}")

# ── Final summary printout ────────────────────────────────────────────────────
print("\n" + "═" * 60)
print(" CNN + BiLSTM + Attention — Final Results Summary")
print("═" * 60)
print(f"  Attention type        : {ATTENTION_TYPE}")
print(f"  Total parameters      : {model.count_params():,}")
print(f"  Training epochs ran   : {len(history.history[loss_key])} / {EPOCHS}")
print(f"  Best validation acc   : {max(history.history[val_acc_key]):.4f}")
print(f"  Test accuracy         : {test_acc:.4f}")
print(f"  Test loss             : {test_loss:.4f}")
print(f"  Macro F1              : {report_dict.get('macro avg', {}).get('f1-score', float('nan')):.4f}")
print("═" * 60)
print("\nPaper reference (A-CBLN, Wu et al. 2023):")
print("  Architecture   : 2D CNN → 2× BiLSTM(8) → Attention → Dense(16)")
print("  Reported acc   : 95.05%  on 7-class handwashing gestures")
print("═" * 60)